In [19]:
import json
import re
import pandas as pd

In [20]:
# Define paths to the downloaded dataset files
BUSINESS_JSON_PATH = "../data/yelp_academic_dataset_business.json"
REVIEW_JSON_PATH = "../data/yelp_academic_dataset_review.json"
OUTPUT_CSV_PATH = "../data/processed_allergy_hazard_dataset.csv"

**Step 1:** Filtering Food & Restaurant Businesses

In [21]:
# Extract a high-lookup set of food and restaurant business IDs
eligible_business_ids = set()
with open(BUSINESS_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        biz = json.loads(line)
        categories = biz.get("categories")
        if categories and ("Restaurants" in categories or "Food" in categories):
            eligible_business_ids.add(biz["business_id"])

print(f"Found {len(eligible_business_ids)} food/restaurant businesses.")

Found 64616 food/restaurant businesses.


**Step 2:** Streaming and Filtering Reviews

In [22]:
allergy_keywords = re.compile(
    r"\b(allergy|allergic|celiac|anaphylactic|epipen|epi-pen|cross-contamination|glutened|contamination|hospital|sick|poisoning)\b",
    re.IGNORECASE,
)

hazard_reviews = []
benign_reviews = []

# Targets to create a robust, balanced dataset for your 3-week sprint
MAX_HAZARDS = 1500
MAX_BENIGN = 6000

with open(REVIEW_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        # Stop early if both buckets are full to save time and RAM
        if len(hazard_reviews) >= MAX_HAZARDS and len(benign_reviews) >= MAX_BENIGN:
            break

        review = json.loads(line)
        if review["business_id"] not in eligible_business_ids:
            continue

        text = review.get("text", "")
        word_count = len(text.split())
        if word_count < 5 or word_count > 800:
            continue

        # Clean text boundaries
        cleaned_text = text.replace("\n", " ").replace("\t", " ")
        cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

        is_allergy_related = bool(allergy_keywords.search(cleaned_text))

        # Define a true hazard: keyword present AND low rating (bad experience)
        is_hazard = 1 if (is_allergy_related and review["stars"] <= 3) else 0

        # Feature engineering metrics for the baseline
        review_features = {
            "stars": review["stars"],
            "useful": review["useful"],
            "funny": review["funny"],
            "cool": review["cool"],
            "text": cleaned_text,
            "word_count": word_count,
            "char_count": len(cleaned_text),
            "exclamation_count": cleaned_text.count("!"),
            "is_hazard": is_hazard,
        }

        # Populate buckets conditionally based on limits
        if is_hazard == 1:
            if len(hazard_reviews) < MAX_HAZARDS:
                hazard_reviews.append(review_features)
        else:
            if len(benign_reviews) < MAX_BENIGN:
                benign_reviews.append(review_features)



**Step 3:** Saving Processed Dataset to CSV

In [27]:
# Combine, shuffle, and save the newly balanced dataset
balanced_list = hazard_reviews + benign_reviews
df = pd.DataFrame(balanced_list)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- Balanced Dataset Summary ---")
print(df["is_hazard"].value_counts())

df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
print(f"\nNew balanced dataset saved to: {OUTPUT_CSV_PATH}")


--- Balanced Dataset Summary ---
is_hazard
0    6000
1    1500
Name: count, dtype: int64
/Users/iriskronfeld/Library/CloudStorage/OneDrive-mail.tau.ac.il/NLP/food_saftey_compass/data/processed_allergy_hazard_dataset.csv

New balanced dataset saved to: /Users/iriskronfeld/Library/CloudStorage/OneDrive-mail.tau.ac.il/NLP/food_saftey_compass/data/processed_allergy_hazard_dataset.csv


In [24]:
df.head(n=100)  # Display the first few rows of the processed dataset for verification

,stars,useful,funny,cool,text,word_count,char_count,exclamation_count,is_hazard
0,3.0,3,2,0,"My wife and I go there at least once a week, t...",538,2847,1,1
1,4.0,0,0,0,We came to the District Tavern for a quick bit...,172,958,5,0
2,1.0,3,1,0,I'm not sure what changed over the last year a...,113,665,0,0
3,5.0,0,0,0,Literally the best meal I have ever eaten. I w...,126,651,0,0
4,5.0,0,0,1,"Awesome ""Fresh"" ingredients, good service, and...",18,107,0,0
...,...,...,...,...,...,...,...,...,...
95,5.0,0,0,0,"Purchased a Groupon for dinner, but turns out ...",128,728,0,0
96,1.0,4,2,0,We arrived a few minutes early for a 7pm reser...,250,1331,0,0
97,4.0,1,0,0,Found out about Slices a few weeks ago and it ...,40,190,0,0
98,5.0,0,0,0,The best Beignet's. Hands down. This great sho...,65,346,3,0
